# Proyecto Integrador 2 · Parte I — Grafo de co-participación
## Tareas 6 y 12: construir el grafo en Spark y describirlo

**Qué hace este notebook**
1. Decide y documenta el **criterio de aristas**: dos proveedores se conectan si **ofertaron en el mismo proceso**.
2. Construye el grafo en **Spark** (la unión de 2.5 millones de postores consigo misma es la parte pesada).
3. Describe el grafo: nodos, aristas, grado, componentes conexas.

**Entrada:** postores del Proyecto 1 y `proveedor_dim` del notebook 1. **Salida:** `grafo_aristas` (Parquet) y 3 archivos para la página web.

**Pregunta de interés que prepara (P3, se responde en las Partes IV y V):** ¿qué actores concentran el mercado y qué comunidades forman? Aquí solo se construye el grafo y se mira cuáles proveedores están más conectados, sin concluir nada todavía.

## 0. Configuración

In [1]:
import os, json
import numpy as np
import pandas as pd

USAR_DRIVE = False   # True si corres en Colab con Drive
RUTA_PARQUET = os.environ.get("P2_PARQUET", "/content/drive/MyDrive/proyecto_integrador_seace/parquet")
RUTA_P2      = os.environ.get("P2_DATOS",   "/content/drive/MyDrive/proyecto_integrador_seace/p2")
WEB          = os.environ.get("P2_WEB",     "/content/drive/MyDrive/proyecto_integrador_seace/web_artefactos/p2")

MIN_PART = 5     # un proveedor es nodo si ofertó en al menos 5 procesos (con menos, casi no hay co-participación que medir)
TOPE     = 30    # procesos con más de 30 postores activos se dejan fuera (ver explicación abajo)
PESO_MIN = 2     # la arista se guarda si los dos proveedores coincidieron en al menos 2 procesos

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(WEB, exist_ok=True)

def guardar_json(nombre, obj):
    with open(f"{WEB}/{nombre}", "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print("guardado:", nombre)

def guardar_csv(nombre, df):
    df.to_csv(f"{WEB}/{nombre}", index=False)
    print("guardado:", nombre, len(df), "filas")

## 1. Spark y lectura

In [2]:
# Solo Colab: !pip install -q pyspark==3.5.3
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[*]").appName("p2_grafo")
         .config("spark.driver.memory", "4g").config("spark.sql.shuffle.partitions", "16").getOrCreate())

dim  = spark.read.parquet(f"{RUTA_P2}/proveedor_dim").select("proveedor_idx", "ruc")
post = spark.read.parquet(f"{RUTA_PARQUET}/postores").select("release_id", "ruc")

# un proveedor por proceso (sin repetidos); al unir con proveedor_dim se van los consorcios (ids de 7 dígitos)
p = post.join(dim, "ruc").select("release_id", "proveedor_idx").distinct().cache()
print("participaciones (proceso, proveedor):", p.count())

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=46287 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=
Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=46287 -Dhttp.nonProxyHosts=localhost|127.0.0.1

26/10/27 23:33:43 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/10/27 23:33:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 23:33:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


participaciones (proceso, proveedor): 2371613


## 2. Criterio de aristas y por qué se filtra

**Criterio:** hay una arista entre A y B si **los dos presentaron oferta en el mismo proceso**. El **peso** es el número de procesos en que coincidieron.

**Por qué postores y no adjudicaciones:** una adjudicación tiene un solo ganador (a veces varios por lotes). Con los postores vemos **con quién compite cada proveedor**, aunque pierda. Además hay adjudicaciones sin contrato vinculado; los postores no tienen ese problema.

**El problema del tamaño:** un proceso con *n* postores genera *n·(n−1)/2* pares. Un proceso con 300 postores genera 44,850 pares. Dos filtros lo controlan:
- **Nodos:** solo proveedores con al menos `MIN_PART` participaciones.
- **Tope:** procesos con más de `TOPE` postores activos se omiten (sale demasiado caro y la señal es débil: con tantos postores, coincidir dice poco). El costo es dejar fuera una parte de los procesos; se mide abajo.

In [3]:
npart = p.groupBy("proveedor_idx").agg(F.count("*").alias("n_part"))
nodos_ok = npart.filter(F.col("n_part") >= MIN_PART).select("proveedor_idx")
p = p.join(F.broadcast(nodos_ok), "proveedor_idx").cache()
print(f"proveedores candidatos a nodo (≥ {MIN_PART} participaciones):", nodos_ok.count())

tam = p.groupBy("release_id").agg(F.count("*").alias("postores_activos")).cache()
filas = []
for t in [10, 15, 20, 30, 50, None]:
    s = tam if t is None else tam.filter(F.col("postores_activos") <= t)
    r = s.agg(F.count("*").alias("procesos"), F.sum(F.col("postores_activos") * (F.col("postores_activos") - 1) / 2).alias("pares")).first()
    filas.append({"tope": "sin tope" if t is None else t, "procesos": r["procesos"], "pares_posibles": int(r["pares"])})
tabla_tope = pd.DataFrame(filas)
tabla_tope["pct_procesos"] = (100 * tabla_tope.procesos / tabla_tope.procesos.iloc[-1]).round(1)
tabla_tope

26/10/27 23:33:58 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


proveedores candidatos a nodo (≥ 5 participaciones): 43115


,tope,procesos,pares_posibles,pct_procesos
0,10,97402,1265750,56.4
1,15,123859,3298718,71.7
2,20,140370,5787652,81.3
3,30,156783,10691800,90.8
4,50,167958,18835325,97.3
5,sin tope,172705,31422554,100.0


Con `TOPE = 30` quedan alrededor de 9 de cada 10 procesos y los pares bajan de unos 31 millones a unos 11 millones. Es el punto en que el cálculo cabe en una máquina local sin perder la mayoría de los procesos.

## 3. Construcción en Spark

In [4]:
procesos_ok = tam.filter(F.col("postores_activos") <= TOPE).select("release_id")
pf = p.join(procesos_ok, "release_id")

a, b = pf.alias("a"), pf.alias("b")
aristas = (a.join(b, (F.col("a.release_id") == F.col("b.release_id")) & (F.col("a.proveedor_idx") < F.col("b.proveedor_idx")))
            .groupBy(F.col("a.proveedor_idx").alias("prov_a"), F.col("b.proveedor_idx").alias("prov_b"))
            .agg(F.count("*").alias("peso")))

# se guarda todo (peso ≥ 1) para poder mostrar qué pasa con otros pesos mínimos
aristas.write.mode("overwrite").parquet(f"{RUTA_P2}/grafo_aristas_peso1")
todas = spark.read.parquet(f"{RUTA_P2}/grafo_aristas_peso1")
print("aristas con peso ≥ 1:", todas.count())

final = todas.filter(F.col("peso") >= PESO_MIN)
final.coalesce(1).write.mode("overwrite").parquet(f"{RUTA_P2}/grafo_aristas")
print(f"aristas con peso ≥ {PESO_MIN} (grafo_aristas):", final.count())

aristas con peso ≥ 1: 3803302


aristas con peso ≥ 2 (grafo_aristas): 1482754


## 4. Tarea 12 — Describir el grafo
Pasamos las aristas a pandas (caben en memoria) para contar componentes conexas con `scipy`.
- **Grado** de un nodo = con cuántos proveedores está conectado.
- **Componente conexa** = grupo de nodos que se alcanzan entre sí por aristas. Si una componente tiene casi todos los nodos, el grafo es "un solo bloque".

In [5]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

e = pd.read_parquet(f"{RUTA_P2}/grafo_aristas_peso1")
print("aristas peso ≥ 1 en memoria:", len(e), "|", round(e.memory_usage().sum()/1e6), "MB")

def describir(df):
    nodos = np.unique(np.concatenate([df.prov_a.values, df.prov_b.values]))
    pos = {n: i for i, n in enumerate(nodos)}
    ia, ib = df.prov_a.map(pos).values, df.prov_b.map(pos).values
    m = coo_matrix((np.ones(len(df)), (ia, ib)), shape=(len(nodos), len(nodos)))
    n_comp, etiqueta = connected_components(m, directed=False)
    tam_comp = np.bincount(etiqueta)
    grado = np.bincount(np.concatenate([ia, ib]), minlength=len(nodos))
    return {"nodos": len(nodos), "aristas": len(df), "componentes": int(n_comp),
            "mayor_componente": int(tam_comp.max()), "pct_mayor_componente": round(100 * tam_comp.max() / len(nodos), 1),
            "grado_medio": round(float(grado.mean()), 1), "grado_mediano": float(np.median(grado)), "grado_max": int(grado.max())}, nodos, grado

filas = []
for k in range(1, 11):
    d, _, _ = describir(e[e.peso >= k])
    d["peso_minimo"] = k
    filas.append(d)
umbrales = pd.DataFrame(filas)[["peso_minimo", "nodos", "aristas", "componentes", "mayor_componente", "pct_mayor_componente", "grado_medio", "grado_mediano", "grado_max"]]
umbrales

aristas peso ≥ 1 en memoria: 3803302 | 61 MB


,peso_minimo,nodos,aristas,componentes,mayor_componente,pct_mayor_componente,grado_medio,grado_mediano,grado_max
0,1,42081,3803302,19,41981,99.8,180.8,88.0,17446
1,2,40106,1482754,100,39798,99.2,73.9,29.0,10705
2,3,36466,844842,245,35787,98.1,46.3,16.0,7325
3,4,31881,568855,357,30905,96.9,35.7,11.0,5380
4,5,27406,417348,381,26372,96.2,30.5,10.0,4196
5,6,23632,324046,340,22677,96.0,27.4,9.0,3408
6,7,20739,261457,304,19918,96.0,25.2,8.0,2841
7,8,18577,217261,301,17792,95.8,23.4,8.0,2407
8,9,16815,183680,265,16133,95.9,21.8,7.0,2075
9,10,15343,157802,246,14708,95.9,20.6,7.0,1813


In [6]:
g, nodos, grado = describir(e[e.peso >= PESO_MIN])
print(g)

# distribución del grado en tramos (escala logarítmica)
cortes = [1, 2, 3, 5, 10, 20, 50, 100, 200, 500, 1000, 10**6]
tramo = pd.cut(grado, bins=cortes, right=False)
dist = pd.Series(tramo).value_counts().sort_index().reset_index()
dist.columns = ["tramo", "n_nodos"]
dist["grado_desde"] = [int(t.left) for t in dist.tramo]
dist["grado_hasta"] = [int(t.right) - 1 if t.right < 10**6 else None for t in dist.tramo]
dist = dist[["grado_desde", "grado_hasta", "n_nodos"]]
dist

{'nodos': 40106, 'aristas': 1482754, 'componentes': 100, 'mayor_componente': 39798, 'pct_mayor_componente': np.float64(99.2), 'grado_medio': 73.9, 'grado_mediano': 29.0, 'grado_max': 10705}


,grado_desde,grado_hasta,n_nodos
0,1,1.0,929
1,2,2.0,1097
2,3,4.0,1969
3,5,9.0,3945
4,10,19.0,6775
5,20,49.0,11637
6,50,99.0,6682
7,100,199.0,4266
8,200,499.0,2150
9,500,999.0,447


In [7]:
# Los 10 proveedores más conectados (solo para mirar; no es una conclusión)
dim_pd = spark.read.parquet(f"{RUTA_P2}/proveedor_dim").toPandas().set_index("proveedor_idx")
sub = e[e.peso >= PESO_MIN]
peso_por_nodo = pd.concat([sub.groupby("prov_a").peso.sum(), sub.groupby("prov_b").peso.sum()]).groupby(level=0).sum()
top = pd.DataFrame({"proveedor_idx": nodos, "grado": grado}).sort_values("grado", ascending=False).head(10)
top["peso_total"] = top.proveedor_idx.map(peso_por_nodo).astype(int)
top["ruc"] = top.proveedor_idx.map(dim_pd.ruc); top["nombre"] = top.proveedor_idx.map(dim_pd.nombre)
top[["nombre", "ruc", "grado", "peso_total"]]

,nombre,ruc,grado,peso_total
36848,TVT QONTRATISTAS E.I.R.L.,20611499591,10705,96119
29063,INVERSIONES GENERALES LAURA ROCA S.A.C.,20605753907,10083,91215
38631,OCAMPER S.A.C.,20613035398,8114,70111
38205,CONSTRUCAPRIGLE E.I.R.L.,20612573621,7868,65254
39366,C.L.D. CONTRATISTAS GENERALES S.A.C.,20613921002,6819,55108
26111,TECNICAS DE INGENIERIA Y CONTRATOS S.A.C. - TE...,20603425104,6359,65083
16868,CRISAMYR SOCIEDAD ANONIMA CERRADA,20542513439,5986,51502
8256,J.C.A. CONTRATISTAS GENERALES S.R.LTDA.,20274419637,5617,56555
30955,COMERCIO E INVERSIONES ANCCAS S.A.C.,20607396516,5256,43624
34373,INVERSIONES Y EDIFICACIONES JYR S.A.C.,20609931311,4956,44582


### Lectura
- El número de aristas **cae muy rápido** al exigir más coincidencias: la mayoría de los pares coincidió en un solo proceso. Con `PESO_MIN = 2` quedan las coincidencias repetidas, que son las que sugieren un mismo mercado.
- **Casi todos los nodos quedan en una sola componente conexa** (99% con `PESO_MIN = 2` y todavía 96% con peso 10). El grafo es un bloque grande y no islas separadas; separar mercados dentro de él será trabajo de las comunidades (Parte V).
- Los proveedores con más conexiones son los que **ofertan en muchísimos procesos**; que tengan muchas conexiones **no prueba nada**, solo que participan mucho. Se usan como punto de partida para PageRank y comunidades (Partes IV y V).

## 5. Archivos para la página web
**Datos personales:** igual que en el Proyecto 1, el RUC que no empieza con 20 se muestra enmascarado (`10283****79`) y el nombre de una persona natural (RUC 10, 15, 16 o 17) se reemplaza por "Persona natural".

In [8]:
def enmascarar(ruc):
    r = str(ruc); return r if r.startswith("20") else r[:5] + "****" + r[-2:]
def nombre_visible(ruc, nombre):
    r = str(ruc); return f"Persona natural (RUC {enmascarar(r)})" if r[:2] in ("10", "15", "16", "17") else nombre
top_web = top[["nombre", "ruc", "grado", "peso_total"]].copy()
top_web["nombre"] = [nombre_visible(r, n) for r, n in zip(top_web.ruc, top_web.nombre)]
top_web["ruc"] = top_web.ruc.map(enmascarar)

guardar_csv("grafo_umbrales.csv", umbrales)
guardar_csv("grafo_grado.csv", dist)
guardar_json("grafo_resumen.json", {
    "criterio": "Arista entre dos proveedores que ofertaron en el mismo proceso; peso = procesos en común.",
    "parametros": {"min_participaciones": MIN_PART, "tope_postores": TOPE, "peso_minimo": PESO_MIN},
    "tabla_tope": tabla_tope.to_dict("records"),
    "grafo_final": g,
    "top_conectados": top_web.to_dict("records"),
})
spark.stop()

guardado: grafo_umbrales.csv 10 filas
guardado: grafo_grado.csv 11 filas
guardado: grafo_resumen.json
